# 🚗 Classical Computer Vision — Vehicle Detection & Counting

This workshop covers **classical (non-DL) Computer Vision** methods applied to
aerial vehicle detection in the [**COWC dataset**](https://gdo152.llnl.gov/cowc/)
(Cars Overhead With Context).

### Why classical CV?
Before deep learning, every vision pipeline was built from explicit image-processing
steps. Understanding them helps you:
- debug DL models (features, artifacts, preprocessing)
- work on low-resource / embedded systems
- build fast baselines before training a network

### Pipeline overview

```
Raw image
   ↓  1. Preprocessing      (noise removal, contrast, color spaces)
   ↓  2. Thresholding        (separate foreground from background)
   ↓  3. Morphological ops   (clean up binary masks)
   ↓  4. Contour detection   (find blobs → count cars)
   ↓  5. HOG features        (gradient-based descriptors)
   ↓  6. HOG + SVM           (patch classifier: car / no-car)
   ↓  7. Sliding window + NMS (full-image detection)
   ↓  8. Evaluation & submission
```

**Dataset:** COWC — Potsdam aerial imagery, 15 cm/pixel resolution.  
**Task:** Count the number of vehicles in each image patch.


---
## 📦 Part 0. Imports

In [1]:
import os
import cv2
import glob
import random
import urllib.request
import tarfile
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

from PIL import Image
from skimage.feature import hog
from skimage import exposure

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle

from scipy import ndimage
import json

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ All imports OK")
print(f"   OpenCV  : {cv2.__version__}")
print(f"   NumPy   : {np.__version__}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Users\Ro

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

---
## 🛰️ Part 1. COWC Dataset

**COWC** (Cars Overhead With Context) contains aerial images from 6 cities
at 15 cm/pixel resolution. We use the pre-cut **64×64 px patches**:

| Split | Positive (car) | Negative (no car) |
|:-----:|:--------------:|:-----------------:|
| train | ~32 000        | ~32 000           |
| test  | ~8 000         | ~8 000            |

Each patch is labelled `1` (car visible) or `0` (no car).

> **For the Kaggle competition** you will receive larger tiles from the Potsdam
> scene and must predict the **car count** per tile.


In [3]:
# ── Download COWC 64×64 patches ──────────────────────────────────────────────
DATA_DIR = Path("cowc_data")
DATA_DIR.mkdir(exist_ok=True)

COWC_URL = (
    "https://gdo152.llnl.gov/cowc/download/"
    "patched-variance-and-perspective/COWC_patch_POTSDAM_3DTop_0064.tar.gz"
)

archive = DATA_DIR / "cowc_patches.tar.gz"

if not archive.exists():
    print("Downloading COWC patches (~600 MB) …")
    urllib.request.urlretrieve(COWC_URL, archive)
    print("✅ Download complete")
else:
    print("✅ Archive already present")

# Extract
extract_dir = DATA_DIR / "patches"
if not extract_dir.exists():
    print("Extracting …")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(extract_dir)
    print("✅ Extracted")
else:
    print("✅ Already extracted")

print("\nDirectory structure:")
for p in sorted(extract_dir.iterdir())[:8]:
    print(f"  {p.name}")


NameError: name 'Path' is not defined

In [ ]:
# ── Parse labels from filenames ──────────────────────────────────────────────
# COWC filename convention:
#   POTSDAM_3DTop_xxxxxx_CAR_1.png   → label = 1
#   POTSDAM_3DTop_xxxxxx_NOCAR_0.png → label = 0

def load_cowc_patches(root, max_per_class=5000):
    """Load patch paths and labels from COWC directory."""
    pos, neg = [], []
    for p in Path(root).rglob("*.png"):
        if "_CAR_" in p.name:
            pos.append((str(p), 1))
        elif "_NOCAR_" in p.name or "NO_CAR" in p.name:
            neg.append((str(p), 0))

    # Balance classes
    random.shuffle(pos); random.shuffle(neg)
    pos = pos[:max_per_class]
    neg = neg[:max_per_class]
    all_samples = pos + neg
    random.shuffle(all_samples)
    paths, labels = zip(*all_samples)
    return list(paths), list(labels)


all_paths, all_labels = load_cowc_patches(extract_dir, max_per_class=6000)

# Train / test split (80/20)
split = int(0.8 * len(all_paths))
train_paths, train_labels = all_paths[:split], all_labels[:split]
test_paths,  test_labels  = all_paths[split:], all_labels[split:]

print(f"Total  : {len(all_paths):,} patches")
print(f"Train  : {len(train_paths):,}  (cars: {sum(train_labels):,})")
print(f"Test   : {len(test_paths):,}   (cars: {sum(test_labels):,})")


In [ ]:
# ── EDA: visualise random patches ────────────────────────────────────────────
def show_patches(paths, labels, n=8, title=""):
    fig, axes = plt.subplots(2, n//2, figsize=(2.5*(n//2), 5))
    axes = axes.ravel()
    car_paths   = [p for p, l in zip(paths, labels) if l == 1][:n//2]
    nocar_paths = [p for p, l in zip(paths, labels) if l == 0][:n//2]
    for i, (p, lbl) in enumerate(zip(car_paths + nocar_paths,
                                      [1]*(n//2) + [0]*(n//2))):
        img = cv2.imread(p)[:, :, ::-1]
        axes[i].imshow(img)
        color = "#2ecc71" if lbl == 1 else "#e74c3c"
        axes[i].set_title("CAR" if lbl else "NO CAR",
                           color=color, fontweight="bold", fontsize=9)
        axes[i].axis("off")
    plt.suptitle(title or "Sample patches", fontsize=12)
    plt.tight_layout(); plt.show()

show_patches(train_paths, train_labels, n=8, title="Sample: car vs no-car patches")


In [ ]:
# ── EDA: pixel intensity statistics ──────────────────────────────────────────
def load_gray(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)

sample_car   = [load_gray(p) for p, l in zip(train_paths, train_labels)
                if l == 1][:200]
sample_nocar = [load_gray(p) for p, l in zip(train_paths, train_labels)
                if l == 0][:200]

mean_car   = np.mean([img.mean() for img in sample_car])
mean_nocar = np.mean([img.mean() for img in sample_nocar])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Average image per class
axes[0].imshow(np.mean(sample_car,   axis=0).astype(np.uint8), cmap="gray")
axes[0].set_title(f"Mean CAR patch  (μ={mean_car:.1f})", fontsize=10)
axes[0].axis("off")

axes[1].imshow(np.mean(sample_nocar, axis=0).astype(np.uint8), cmap="gray")
axes[1].set_title(f"Mean NO-CAR patch (μ={mean_nocar:.1f})", fontsize=10)
axes[1].axis("off")

# Histogram
for imgs, lbl, color in [(sample_car,   "CAR",    "#2ecc71"),
                          (sample_nocar, "NO CAR", "#e74c3c")]:
    all_px = np.concatenate([i.ravel() for i in imgs])
    axes[2].hist(all_px, bins=64, alpha=0.5, label=lbl, color=color, density=True)
axes[2].set_title("Pixel intensity distribution")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("EDA — Patch Statistics", fontsize=13)
plt.tight_layout(); plt.show()


---
## 🔧 Part 2. Preprocessing

Classical CV is very sensitive to image quality. Before extracting any features
we apply a standard preprocessing chain.

### Tools:
| Technique | Purpose | OpenCV function |
|:---------:|:-------:|:---------------:|
| Gaussian blur | Remove high-freq noise | `cv2.GaussianBlur` |
| CLAHE | Improve local contrast | `cv2.createCLAHE` |
| Color space conversion | Separate illumination/color | `cv2.cvtColor` |
| Histogram equalisation | Global contrast boost | `cv2.equalizeHist` |


In [ ]:
def preprocess(img_bgr, blur_ksize=3, clahe=True):
    """
    Standard preprocessing pipeline.

    Parameters
    ----------
    img_bgr  : np.ndarray  BGR image (as read by cv2.imread)
    blur_ksize : int       Gaussian kernel size (must be odd)
    clahe      : bool      Apply CLAHE contrast enhancement

    Returns
    -------
    gray_proc : np.ndarray  Preprocessed grayscale image (uint8)
    """
    # 1. Convert to grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian blur — remove sensor noise
    gray = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)

    # 3. CLAHE — Contrast Limited Adaptive Histogram Equalisation
    #    Improves local contrast without over-amplifying noise
    if clahe:
        clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        gray = clahe_obj.apply(gray)

    return gray


# ── Visualise preprocessing stages ───────────────────────────────────────────
sample_path = train_paths[0]
img_bgr     = cv2.imread(sample_path)
gray_raw    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_blur   = cv2.GaussianBlur(gray_raw, (3, 3), 0)
clahe_obj   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
gray_clahe  = clahe_obj.apply(gray_blur)

stages = [
    (img_bgr[:, :, ::-1], "Original (RGB)"),
    (gray_raw,             "Grayscale"),
    (gray_blur,            "Gaussian Blur (k=3)"),
    (gray_clahe,           "CLAHE"),
]

fig, axes = plt.subplots(1, len(stages), figsize=(14, 3))
for ax, (img, title) in zip(axes, stages):
    cmap = None if img.ndim == 3 else "gray"
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10); ax.axis("off")
plt.suptitle("Preprocessing stages", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── Color spaces: what can we learn beyond grayscale? ────────────────────────
img_bgr  = cv2.imread(train_paths[1])
img_rgb  = img_bgr[:, :, ::-1]
img_hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

channels = [
    (img_rgb[:,:,0],   "R"),
    (img_rgb[:,:,1],   "G"),
    (img_rgb[:,:,2],   "B"),
    (img_hsv[:,:,0],   "H (hue)"),
    (img_hsv[:,:,1],   "S (saturation)"),
    (img_hsv[:,:,2],   "V (value)"),
    (img_lab[:,:,0],   "L (lightness)"),
    (img_lab[:,:,1],   "A (green↔red)"),
    (img_lab[:,:,2],   "B (blue↔yellow)"),
]

fig, axes = plt.subplots(2, len(channels)//2 + 1, figsize=(18, 6))
axes = axes.ravel()
axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis("off")
for ax, (ch, name) in zip(axes[1:], channels):
    ax.imshow(ch, cmap="gray")
    ax.set_title(name, fontsize=9); ax.axis("off")
for ax in axes[len(channels)+1:]:
    ax.set_visible(False)
plt.suptitle("Color space channels", fontsize=12)
plt.tight_layout(); plt.show()


---
## 🎭 Part 3. Thresholding & Morphological Operations

**Thresholding** converts a grayscale image to a binary mask:
pixels brighter than a threshold → foreground (1), rest → background (0).

**Morphological operations** refine binary masks:

| Operation | Effect | Use case |
|:---------:|:------:|:--------:|
| Erosion | Shrink white regions | Remove tiny noise dots |
| Dilation | Expand white regions | Fill small holes |
| Opening (erosion→dilation) | Remove small blobs | Clean noisy mask |
| Closing (dilation→erosion) | Fill small holes | Connect broken shapes |


In [ ]:
# ── Thresholding methods compared ────────────────────────────────────────────
img_bgr  = cv2.imread(train_paths[0])
gray     = preprocess(img_bgr)

_, thresh_global   = cv2.threshold(gray, 127,  255, cv2.THRESH_BINARY)
_, thresh_otsu     = cv2.threshold(gray, 0,    255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
thresh_adaptive    = cv2.adaptiveThreshold(gray, 255,
                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                         cv2.THRESH_BINARY, 11, 2)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (img, title) in zip(axes, [
    (gray,           f"Preprocessed"),
    (thresh_global,  "Global thresh (t=127)"),
    (thresh_otsu,    "Otsu auto-threshold"),
    (thresh_adaptive,"Adaptive (Gaussian)"),
]):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=9); ax.axis("off")
plt.suptitle("Thresholding methods", fontsize=12)
plt.tight_layout(); plt.show()

otsu_val = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
print(f"Otsu optimal threshold: {otsu_val:.0f}")


In [ ]:
# ── Morphological operations ─────────────────────────────────────────────────
kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

_, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

eroded  = cv2.erode(mask,  kernel3, iterations=1)
dilated = cv2.dilate(mask, kernel3, iterations=1)
opened  = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel3)
closed  = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel5)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, (img, title) in zip(axes, [
    (mask,    "Otsu mask"),
    (eroded,  "Erosion (k=3)"),
    (dilated, "Dilation (k=3)"),
    (opened,  "Opening (k=3)"),
    (closed,  "Closing (k=5)"),
]):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=9); ax.axis("off")
plt.suptitle("Morphological operations", fontsize=12)
plt.tight_layout(); plt.show()


---
## 🔵 Part 4. Contour Detection — Simple Car Counter

A **contour** is a curve joining all continuous points along a boundary
with the same intensity. We find blobs in the binary mask and filter them
by area and shape to count cars.

### Key filters:
- **Area** — cars occupy ~200–2000 px² at 15 cm/px resolution
- **Aspect ratio** — cars are roughly square/rectangular (ratio 0.3–3.0)
- **Extent** — ratio of contour area to bounding-box area


In [ ]:
def contour_count(img_bgr,
                  blur_k=3,
                  min_area=150, max_area=2500,
                  min_aspect=0.3, max_aspect=3.0,
                  morph_open_k=3, morph_close_k=5):
    """
    Count cars using thresholding + morphology + contour filtering.

    Returns
    -------
    count   : int            number of detected cars
    vis     : np.ndarray     BGR visualisation image
    contours_kept : list     kept contours
    """
    gray = preprocess(img_bgr, blur_ksize=blur_k)

    # Otsu threshold
    _, mask = cv2.threshold(gray, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological clean-up
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                        (morph_open_k, morph_open_k))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                        (morph_close_k, morph_close_k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    # Filter by shape
    kept = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = w / (h + 1e-6)
        if aspect < min_aspect or aspect > max_aspect:
            continue
        kept.append(cnt)

    # Draw
    vis = img_bgr.copy()
    cv2.drawContours(vis, kept, -1, (0, 255, 0), 1)
    for cnt in kept:
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 200, 255), 1)

    return len(kept), vis, kept


# ── Demo on a few patches ────────────────────────────────────────────────────
demo_paths  = [p for p, l in zip(train_paths, train_labels) if l == 1][:6]
fig, axes   = plt.subplots(2, 6, figsize=(18, 6))

for col, path in enumerate(demo_paths):
    img = cv2.imread(path)
    cnt, vis, _ = contour_count(img)

    axes[0, col].imshow(img[:, :, ::-1])
    axes[0, col].set_title("Original", fontsize=8); axes[0, col].axis("off")

    axes[1, col].imshow(vis[:, :, ::-1])
    axes[1, col].set_title(f"Detected: {cnt}", fontsize=8,
                            color="#2ecc71" if cnt > 0 else "#e74c3c")
    axes[1, col].axis("off")

plt.suptitle("Contour-based car detection on patches", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── Quick evaluation: contour counter accuracy ────────────────────────────────
EVAL_N = 500
sample_paths  = train_paths[:EVAL_N]
sample_labels = train_labels[:EVAL_N]

preds = []
for path in sample_paths:
    img = cv2.imread(path)
    count, _, _ = contour_count(img)
    preds.append(1 if count > 0 else 0)

preds   = np.array(preds)
labels  = np.array(sample_labels)

tp = int(((preds == 1) & (labels == 1)).sum())
fp = int(((preds == 1) & (labels == 0)).sum())
fn = int(((preds == 0) & (labels == 1)).sum())
tn = int(((preds == 0) & (labels == 0)).sum())

precision = tp / (tp + fp + 1e-9)
recall    = tp / (tp + fn + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
accuracy  = (tp + tn) / len(labels)

print("Contour counter — patch classification results")
print(f"  Accuracy  : {accuracy*100:.1f}%")
print(f"  Precision : {precision*100:.1f}%")
print(f"  Recall    : {recall*100:.1f}%")
print(f"  F1        : {f1*100:.1f}%")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")


### ✏️ Exercise 1 — Tune the contour parameters

The default parameters may not be optimal. Try to improve the F1 score
by changing the filters in `contour_count`.

**Hints:**
- `min_area` / `max_area` — what is the typical pixel area of a car at 15 cm/px?
- `morph_open_k` — larger kernel removes more noise but also small cars
- Try `MORPH_GRADIENT` instead of thresholding to detect edges

Fill in your best parameters below:


In [ ]:
# TODO: find better parameters for contour_count
# Evaluate on the same 500 patches

MY_PARAMS = dict(
    blur_k       = 3,      # ← try 3, 5
    min_area     = 150,    # ← try 100–300
    max_area     = 2500,   # ← try 1500–4000
    min_aspect   = 0.3,    # ← try 0.2–0.5
    max_aspect   = 3.0,    # ← try 2.0–4.0
    morph_open_k = 3,      # ← try 2–5
    morph_close_k= 5,      # ← try 3–7
)

preds_ex1 = []
for path in sample_paths:
    img = cv2.imread(path)
    count, _, _ = contour_count(img, **MY_PARAMS)
    preds_ex1.append(1 if count > 0 else 0)

preds_ex1 = np.array(preds_ex1)
tp = int(((preds_ex1 == 1) & (labels == 1)).sum())
fp = int(((preds_ex1 == 1) & (labels == 0)).sum())
fn = int(((preds_ex1 == 0) & (labels == 1)).sum())
tn = int(((preds_ex1 == 0) & (labels == 0)).sum())

precision = tp / (tp + fp + 1e-9)
recall    = tp / (tp + fn + 1e-9)
f1_ex1    = 2 * precision * recall / (precision + recall + 1e-9)
accuracy  = (tp + tn) / len(labels)

print(f"Your contour counter:")
print(f"  Accuracy  : {accuracy*100:.1f}%")
print(f"  F1        : {f1_ex1*100:.1f}%")

if f1_ex1 > f1:
    print(f"  ✅ Improvement! +{(f1_ex1-f1)*100:.1f}% F1")
else:
    print(f"  Keep tuning! Baseline F1 was {f1*100:.1f}%")


---
## 📐 Part 5. HOG Features — Histogram of Oriented Gradients

**HOG** (Dalal & Triggs, 2005) is the backbone of classical object detection.  
Instead of raw pixels, it describes the **distribution of gradient directions**.

### How HOG works:

```
Image
 ↓  1. Compute gradients  Gx = ∂I/∂x,  Gy = ∂I/∂y
 ↓  2. Magnitude & angle  M = √(Gx²+Gy²),  θ = arctan(Gy/Gx)
 ↓  3. Divide into cells  (e.g. 8×8 px each)
 ↓  4. Build histogram    9 orientation bins per cell
 ↓  5. Group into blocks  (e.g. 2×2 cells) and L2-normalise
 ↓  HOG vector            ← compact, illumination-invariant descriptor
```

### Key parameters:
| Parameter | Typical value | Effect |
|:---------:|:-------------:|:------:|
| `pixels_per_cell` | (8,8) | Smaller = more detail, larger vector |
| `cells_per_block` | (2,2) | Larger = better normalisation |
| `orientations`    | 9     | More bins = finer angle resolution |


In [ ]:
# ── Visualise HOG on car vs no-car ───────────────────────────────────────────
HOG_PARAMS = dict(
    orientations  = 9,
    pixels_per_cell = (8, 8),
    cells_per_block = (2, 2),
    visualize     = True,
    channel_axis  = None,   # grayscale input
)

car_path   = next(p for p, l in zip(train_paths, train_labels) if l == 1)
nocar_path = next(p for p, l in zip(train_paths, train_labels) if l == 0)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for row, (path, label) in enumerate([(car_path, "CAR"), (nocar_path, "NO CAR")]):
    img_bgr = cv2.imread(path)
    gray    = preprocess(img_bgr)

    fd, hog_img = hog(gray, **HOG_PARAMS)
    hog_vis = exposure.rescale_intensity(hog_img, in_range=(0, 10))

    # Gradient visualisation (Sobel)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)

    axes[row, 0].imshow(img_bgr[:,:,::-1])
    axes[row, 0].set_title(f"{label} — Original"); axes[row, 0].axis("off")

    axes[row, 1].imshow(mag, cmap="hot")
    axes[row, 1].set_title("Gradient magnitude"); axes[row, 1].axis("off")

    axes[row, 2].imshow(hog_vis, cmap="gray")
    axes[row, 2].set_title(f"HOG  (dim={len(fd)})")
    axes[row, 2].axis("off")

plt.suptitle("HOG feature visualisation", fontsize=13)
plt.tight_layout(); plt.show()

# Print feature vector info
fd_car, _ = hog(preprocess(cv2.imread(car_path)), **HOG_PARAMS)
print(f"HOG feature vector length: {len(fd_car)}")
print(f"  = (64/8 - 1) × (64/8 - 1) × 2×2 cells/block × 9 orientations")
print(f"  = 7 × 7 × 4 × 9 = {7*7*4*9}")


In [ ]:
# ── HOG histogram per cell — what do the bins mean? ─────────────────────────
img_bgr = cv2.imread(car_path)
gray    = preprocess(img_bgr)

gx = cv2.Sobel(gray.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(gray.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3)
angles = np.degrees(np.arctan2(gy, gx)) % 180   # 0–180 unsigned
mag    = np.sqrt(gx**2 + gy**2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title("Preprocessed"); axes[0].axis("off")
axes[1].imshow(angles, cmap="hsv"); axes[1].set_title("Gradient angles (°)"); axes[1].axis("off")

# Weighted histogram of angles over the whole patch
hist, bins = np.histogram(angles.ravel(), bins=9, range=(0, 180),
                           weights=mag.ravel())
bin_centers = (bins[:-1] + bins[1:]) / 2
axes[2].bar(bin_centers, hist, width=18, color="steelblue", alpha=0.8)
axes[2].set_xlabel("Gradient orientation (°)")
axes[2].set_ylabel("Weighted count")
axes[2].set_title("Gradient orientation histogram")
axes[2].set_xticks(range(0, 181, 20))
axes[2].grid(alpha=0.3)

plt.suptitle("HOG internals — gradient orientations", fontsize=12)
plt.tight_layout(); plt.show()


---
## 🤖 Part 6. HOG + SVM Classifier

We extract HOG descriptors from all patches and train a **Support Vector
Machine (SVM)** to classify each 64×64 patch as *car* or *no-car*.

### Why SVM with HOG?
- HOG captures shape information that is invariant to illumination
- SVM finds the maximum-margin hyperplane in HOG feature space
- Fast to train (~seconds) and fast to evaluate (~μs per patch)

This was the **state of the art** for pedestrian/vehicle detection until 2012
(ImageNet moment).


In [ ]:
# ── Feature extraction ────────────────────────────────────────────────────────
PATCH_HOG_PARAMS = dict(
    orientations    = 9,
    pixels_per_cell = (8, 8),
    cells_per_block = (2, 2),
    visualize       = False,
    channel_axis    = None,
)

def extract_hog(path):
    """Load patch → preprocess → HOG vector."""
    img = cv2.imread(path)
    gray = preprocess(img, blur_ksize=3, clahe=True)
    return hog(gray, **PATCH_HOG_PARAMS)


print("Extracting HOG features …")
print("  Training set …")
X_train = np.array([extract_hog(p) for p in train_paths])
y_train = np.array(train_labels)

print("  Test set …")
X_test  = np.array([extract_hog(p) for p in test_paths])
y_test  = np.array(test_labels)

print(f"\nFeature matrix shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  Feature vector dim: {X_train.shape[1]}")


In [ ]:
# ── Train SVM ─────────────────────────────────────────────────────────────────
from sklearn.pipeline import make_pipeline

# Pipeline: StandardScaler → LinearSVC
#   StandardScaler: zero mean, unit variance — important for SVM!
#   LinearSVC:      fast, works well with HOG

svm_pipeline = make_pipeline(
    StandardScaler(),
    LinearSVC(C=0.01, max_iter=2000, random_state=SEED)
)

print("Training HOG + LinearSVM …")
svm_pipeline.fit(X_train, y_train)
print("✅ Training complete")

# Evaluate
y_pred_train = svm_pipeline.predict(X_train)
y_pred_test  = svm_pipeline.predict(X_test)

acc_train = (y_pred_train == y_train).mean()
acc_test  = (y_pred_test  == y_test ).mean()

print(f"\nAccuracy  train : {acc_train*100:.1f}%")
print(f"Accuracy  test  : {acc_test*100:.1f}%")
print()
print(classification_report(y_test, y_pred_test,
                             target_names=["NO CAR", "CAR"]))


In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(cm, display_labels=["NO CAR", "CAR"]).plot(ax=axes[0])
axes[0].set_title("HOG + SVM — Confusion Matrix")

# ── Visualise false positives & false negatives ───────────────────────────────
fp_idx = [i for i,(p,t) in enumerate(zip(y_pred_test, y_test)) if p==1 and t==0][:4]
fn_idx = [i for i,(p,t) in enumerate(zip(y_pred_test, y_test)) if p==0 and t==1][:4]

n = min(4, len(fp_idx), len(fn_idx))
if n > 0:
    imgs_fp = [cv2.imread(test_paths[i])[:,:,::-1] for i in fp_idx[:n]]
    imgs_fn = [cv2.imread(test_paths[i])[:,:,::-1] for i in fn_idx[:n]]

    ax = axes[1]
    ax.axis("off")
    ax.set_title("False Positives (top) vs False Negatives (bottom)")

    for col in range(n):
        inset = ax.inset_axes([col/n, 0.5, 1/n-0.02, 0.45])
        inset.imshow(imgs_fp[col]); inset.set_title("FP", color="red", fontsize=8)
        inset.axis("off")

        inset2 = ax.inset_axes([col/n, 0.0, 1/n-0.02, 0.45])
        inset2.imshow(imgs_fn[col]); inset2.set_title("FN", color="orange", fontsize=8)
        inset2.axis("off")

plt.tight_layout(); plt.show()


In [ ]:
# ── Effect of SVM regularisation parameter C ─────────────────────────────────
print("Cross-validating over C values …")
C_values = [0.001, 0.01, 0.1, 1.0, 10.0]
cv_scores = []

for C in C_values:
    pipe = make_pipeline(StandardScaler(),
                         LinearSVC(C=C, max_iter=2000, random_state=SEED))
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="f1", n_jobs=-1)
    cv_scores.append(scores.mean())
    print(f"  C={C:.3f}  F1={scores.mean():.3f} ± {scores.std():.3f}")

best_C = C_values[np.argmax(cv_scores)]
print(f"\nBest C = {best_C}  →  F1 = {max(cv_scores):.3f}")

plt.figure(figsize=(6, 3))
plt.semilogx(C_values, cv_scores, "o-", color="steelblue")
plt.axvline(best_C, linestyle="--", color="red", label=f"Best C={best_C}")
plt.xlabel("C (regularisation)"); plt.ylabel("CV F1 score")
plt.title("SVM regularisation sweep"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# ── Retrain with best C ──────────────────────────────────────────────────────
svm_best = make_pipeline(StandardScaler(),
                         LinearSVC(C=best_C, max_iter=2000, random_state=SEED))
svm_best.fit(X_train, y_train)

y_pred_best = svm_best.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=["NO CAR", "CAR"]))


---
## 🪟 Part 7. Sliding Window + NMS — Full-Image Detection

Patch classifiers work on fixed-size windows. To detect cars in a **large tile**
we slide the classifier window across the image at multiple positions.

### Pipeline:
```
Large tile  (e.g. 512×512)
  ↓  Slide 64×64 window with stride S
  ↓  Extract HOG from each window
  ↓  SVM score → keep windows with score > threshold
  ↓  Non-Maximum Suppression (NMS) — remove overlapping detections
  ↓  Final bounding boxes = detected cars
```

### Non-Maximum Suppression (NMS):
When two overlapping boxes detect the same car, keep only the highest-scoring one.
IoU threshold controls how much overlap is tolerated.

```
IoU = area(A ∩ B) / area(A ∪ B)
If IoU > threshold → suppress the lower-score box
```


In [ ]:
def non_max_suppression(boxes, scores, iou_threshold=0.3):
    """
    Classic Greedy NMS.

    Parameters
    ----------
    boxes  : np.ndarray  shape (N, 4) — [x1, y1, x2, y2]
    scores : np.ndarray  shape (N,)
    iou_threshold : float

    Returns
    -------
    keep : list of indices to keep
    """
    if len(boxes) == 0:
        return []

    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas  = (x2 - x1 + 1) * (y2 - y1 + 1)
    order  = scores.argsort()[::-1]     # highest score first

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        # Compute IoU with remaining boxes
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        inter = np.maximum(0, xx2 - xx1 + 1) * np.maximum(0, yy2 - yy1 + 1)
        iou   = inter / (areas[i] + areas[order[1:]] - inter)

        inds  = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]

    return keep


def sliding_window_detect(tile_bgr, classifier,
                           win_size=64, stride=16,
                           score_threshold=0.0,
                           nms_iou=0.3):
    """
    Run sliding window HOG+SVM detector on a large tile.

    Returns
    -------
    boxes  : np.ndarray (N, 4)  [x1, y1, x2, y2]
    scores : np.ndarray (N,)
    """
    H, W = tile_bgr.shape[:2]
    boxes_list  = []
    scores_list = []

    for y in range(0, H - win_size + 1, stride):
        for x in range(0, W - win_size + 1, stride):
            patch = tile_bgr[y:y+win_size, x:x+win_size]
            gray  = preprocess(patch)
            feat  = hog(gray, **PATCH_HOG_PARAMS).reshape(1, -1)

            # Decision function gives distance from hyperplane (= confidence)
            score = classifier.decision_function(feat)[0]
            if score > score_threshold:
                boxes_list.append([x, y, x+win_size, y+win_size])
                scores_list.append(score)

    if not boxes_list:
        return np.empty((0, 4)), np.empty((0,))

    boxes  = np.array(boxes_list)
    scores = np.array(scores_list)

    # Apply NMS
    keep   = non_max_suppression(boxes, scores, nms_iou)
    return boxes[keep], scores[keep]


In [ ]:
# ── Build a larger test tile by tiling patches ────────────────────────────────
# (We synthesise a 256×256 tile from 16 patches for demonstration)
TILE_GRID = 4   # 4×4 = 16 patches → 256×256 tile

car_patches   = [cv2.imread(p) for p, l in zip(train_paths, train_labels)
                 if l == 1][:TILE_GRID*TILE_GRID//2]
nocar_patches = [cv2.imread(p) for p, l in zip(train_paths, train_labels)
                 if l == 0][:TILE_GRID*TILE_GRID//2]
all_patches   = car_patches + nocar_patches
random.shuffle(all_patches)

rows = []
for r in range(TILE_GRID):
    row_imgs = all_patches[r*TILE_GRID:(r+1)*TILE_GRID]
    rows.append(np.hstack(row_imgs))
tile = np.vstack(rows)

print(f"Synthetic tile size: {tile.shape}")

# ── Run detector ─────────────────────────────────────────────────────────────
boxes, scores = sliding_window_detect(
    tile, svm_best,
    win_size=64, stride=32,
    score_threshold=0.5,
    nms_iou=0.3
)
print(f"Detected {len(boxes)} cars (before NMS: {len(scores)})")

# ── Visualise ─────────────────────────────────────────────────────────────────
vis_tile = tile.copy()
for (x1, y1, x2, y2), score in zip(boxes, scores):
    cv2.rectangle(vis_tile, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis_tile, f"{score:.1f}", (x1, max(y1-4, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0,255,0), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(tile[:,:,::-1]); axes[0].set_title("Tile (original)"); axes[0].axis("off")
axes[1].imshow(vis_tile[:,:,::-1])
axes[1].set_title(f"Sliding window + NMS  ({len(boxes)} detections)")
axes[1].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ── Effect of NMS IoU threshold ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, nms_thr in zip(axes, [0.1, 0.3, 0.6]):
    boxes_tmp, scores_tmp = sliding_window_detect(
        tile, svm_best,
        win_size=64, stride=32,
        score_threshold=0.5,
        nms_iou=nms_thr
    )
    vis = tile.copy()
    for (x1,y1,x2,y2) in boxes_tmp:
        cv2.rectangle(vis, (x1,y1),(x2,y2),(0,255,0),2)
    ax.imshow(vis[:,:,::-1])
    ax.set_title(f"NMS IoU={nms_thr}  →  {len(boxes_tmp)} dets")
    ax.axis("off")

plt.suptitle("Effect of NMS IoU threshold", fontsize=12)
plt.tight_layout(); plt.show()


---
## 📏 Part 8. Evaluation — Comparing Methods

We compare three approaches on the same test patches:
1. **Contour counter** — thresholding + morphology + contour filter
2. **HOG + SVM** — learned patch classifier
3. *(Exercise 2: Watershed)*

### Metrics for patch binary classification:

| Metric | Formula | Interpretation |
|:------:|:-------:|:--------------:|
| Accuracy  | (TP+TN)/N | Overall correctness |
| Precision | TP/(TP+FP) | Of all positives predicted, how many are real? |
| Recall    | TP/(TP+FN) | Of all real positives, how many were found? |
| F1        | 2·P·R/(P+R) | Harmonic mean of precision & recall |


In [ ]:
# ── Compare contour vs HOG+SVM on the full test set ──────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

print("Evaluating on test set …")

# HOG + SVM predictions (already computed)
pred_svm = y_pred_best.copy()

# Contour predictions
pred_contour = []
for path in test_paths:
    img = cv2.imread(path)
    count, _, _ = contour_count(img, **MY_PARAMS)
    pred_contour.append(1 if count > 0 else 0)
pred_contour = np.array(pred_contour)

results_table = {}
for name, preds in [("Contour counter", pred_contour),
                     ("HOG + SVM",       pred_svm)]:
    results_table[name] = {
        "Accuracy"  : accuracy_score(y_test, preds),
        "Precision" : precision_score(y_test, preds, zero_division=0),
        "Recall"    : recall_score(y_test, preds, zero_division=0),
        "F1"        : f1_score(y_test, preds, zero_division=0),
    }

df_results = pd.DataFrame(results_table).T * 100
print(df_results.round(1).to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
df_results.plot(kind="bar", ax=ax, alpha=0.85,
                color=["#3498db","#e74c3c","#2ecc71","#f39c12"])
ax.set_ylabel("Score (%)"); ax.set_title("Method comparison")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout(); plt.show()


### ✏️ Exercise 2 — Watershed Segmentation (Independent)

**Watershed** treats the image as a topographic surface (pixel intensity = altitude)
and floods it from markers. It is excellent at **separating touching objects**.

This is useful when cars are parked close together and the contour method merges them.

Implement `watershed_count(img_bgr)` that:
1. Preprocesses the image
2. Applies Otsu threshold
3. Computes the **distance transform** of the binary mask
4. Finds local maxima as **markers** for watershed
5. Runs `cv2.watershed` and counts the number of regions

**Key functions:**
- `ndimage.distance_transform_edt(mask)` — distance transform
- `ndimage.label(sure_fg)` — label connected components as markers  
- `cv2.watershed(img, markers)` — watershed segmentation

**Hint:** the sure foreground is found by thresholding the distance transform
at `0.5 × max_distance`.


In [ ]:
def watershed_count(img_bgr):
    """
    Count cars using Watershed segmentation.

    TODO: implement this function.

    Returns
    -------
    count : int   number of detected car regions
    vis   : np.ndarray  BGR visualisation
    """
    gray = preprocess(img_bgr)

    # Step 1: Otsu threshold → binary mask
    _, mask = cv2.threshold(gray, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 2: morphological opening to remove noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    sure_bg = cv2.dilate(mask, kernel, iterations=2)

    # TODO Step 3: distance transform
    # dist = ndimage.distance_transform_edt(mask)

    # TODO Step 4: sure foreground = dist > 0.5 * dist.max()
    # sure_fg = (dist > 0.5 * dist.max()).astype(np.uint8) * 255

    # TODO Step 5: unknown region = sure_bg - sure_fg
    # unknown = cv2.subtract(sure_bg, sure_fg)

    # TODO Step 6: label markers
    # _, markers = ndimage.label(sure_fg)
    # markers = markers + 1
    # markers[unknown == 255] = 0

    # TODO Step 7: watershed
    # img_color = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    # markers = cv2.watershed(img_color, markers)
    # count = markers.max() - 1   # subtract background label

    count = 0           # ← replace with your result
    vis   = img_bgr.copy()  # ← replace with your visualisation
    return count, vis


# Quick test (run after implementing)
img_test = cv2.imread(train_paths[0])
count_ws, vis_ws = watershed_count(img_test)
print(f"Watershed count: {count_ws}")


### ✏️ Exercise 3 — Full Comparison

After completing Exercise 2, add Watershed to the comparison table.

Answer the following questions in a markdown cell:
1. Which method has the best F1? Why?
2. In which cases does the contour method fail?
3. What does the SVM learn that the contour method cannot?
4. What are the limitations of sliding window vs modern detectors (YOLO)?


In [ ]:
# TODO: evaluate watershed on the test set and add to results_table
# Uncomment after implementing watershed_count

# pred_watershed = []
# for path in test_paths:
#     img = cv2.imread(path)
#     count, _ = watershed_count(img)
#     pred_watershed.append(1 if count > 0 else 0)
# pred_watershed = np.array(pred_watershed)
#
# results_table["Watershed"] = {
#     "Accuracy"  : accuracy_score(y_test, pred_watershed),
#     "Precision" : precision_score(y_test, pred_watershed, zero_division=0),
#     "Recall"    : recall_score(y_test, pred_watershed, zero_division=0),
#     "F1"        : f1_score(y_test, pred_watershed, zero_division=0),
# }
#
# df_full = pd.DataFrame(results_table).T * 100
# print(df_full.round(1).to_string())


---
## 🏆 Part 9. Kaggle Submission

The Kaggle competition gives you **larger tiles** (cropped from the Potsdam scene).
Your task: predict the **number of cars** in each tile.

We run the sliding-window detector and count surviving bounding boxes after NMS.


In [ ]:
# ── Paths (adjust to match the Kaggle test data location) ────────────────────
KAGGLE_TEST_DIR = Path("kaggle_test")   # ← update this path

def count_cars_in_tile(tile_bgr, classifier,
                       win_size=64, stride=16,
                       score_threshold=0.3,
                       nms_iou=0.3):
    """Returns car count for one tile using sliding window + NMS."""
    boxes, scores = sliding_window_detect(
        tile_bgr, classifier,
        win_size=win_size, stride=stride,
        score_threshold=score_threshold,
        nms_iou=nms_iou
    )
    return len(boxes)


def generate_submission(test_dir, classifier,
                         win_size=64, stride=16,
                         score_threshold=0.3,
                         nms_iou=0.3,
                         output_csv="submission.csv"):
    """
    Run detector on all tiles in test_dir and write submission CSV.

    Expected CSV format:
        image_id,count
        tile_001,5
        tile_002,3
        ...
    """
    records = []
    tile_paths = sorted(Path(test_dir).glob("*.jpg")) +                  sorted(Path(test_dir).glob("*.png"))

    for path in tile_paths:
        tile = cv2.imread(str(path))
        if tile is None:
            continue
        count = count_cars_in_tile(tile, classifier,
                                   win_size=win_size, stride=stride,
                                   score_threshold=score_threshold,
                                   nms_iou=nms_iou)
        records.append({"image_id": path.stem, "count": count})

    df_sub = pd.DataFrame(records)
    df_sub.to_csv(output_csv, index=False)
    print(f"✅ Submission saved: {output_csv}  ({len(df_sub)} tiles)")
    return df_sub


# ── Run (comment out if Kaggle test dir is not available yet) ─────────────────
if KAGGLE_TEST_DIR.exists():
    df_submission = generate_submission(
        KAGGLE_TEST_DIR, svm_best,
        stride=16, score_threshold=0.3, nms_iou=0.3
    )
    print(df_submission.head(10))
    print(f"\nPredicted counts: min={df_submission['count'].min()}, "
          f"max={df_submission['count'].max()}, "
          f"mean={df_submission['count'].mean():.1f}")
else:
    print(f"ℹ️  Kaggle test directory not found: {KAGGLE_TEST_DIR}")
    print("   Update KAGGLE_TEST_DIR after downloading the competition data.")


In [ ]:
# ── Validate submission on our own test patches ──────────────────────────────
# Use patches directly as "tiles" (each is already 64×64)

records_val = []
for path, true_label in zip(test_paths[:200], test_labels[:200]):
    tile = cv2.imread(path)
    count = count_cars_in_tile(tile, svm_best,
                                win_size=64, stride=32,
                                score_threshold=0.2,
                                nms_iou=0.3)
    records_val.append({"true": true_label, "pred_count": count,
                         "pred_binary": 1 if count > 0 else 0})

df_val = pd.DataFrame(records_val)

acc_val = (df_val["true"] == df_val["pred_binary"]).mean()
mae     = (df_val["true"] - df_val["pred_count"]).abs().mean()
print(f"Patch-level validation (200 samples):")
print(f"  Binary accuracy : {acc_val*100:.1f}%")
print(f"  Count MAE       : {mae:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df_val["pred_count"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title("Distribution of predicted counts")
axes[0].set_xlabel("Predicted car count"); axes[0].set_ylabel("# patches")
axes[0].grid(axis="y", alpha=0.3)

pd.crosstab(df_val["true"], df_val["pred_binary"],
            rownames=["True"], colnames=["Predicted"]).plot(
    kind="bar", ax=axes[1], color=["#e74c3c","#2ecc71"], alpha=0.8)
axes[1].set_title("True vs Predicted (binary)"); axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()


---
## 📋 Summary

### What we built — a complete classical CV pipeline:

```
Raw patch (64×64 px)
  ↓  Gaussian blur + CLAHE          → reduce noise, boost contrast
  ↓  Otsu threshold                 → separate foreground/background
  ↓  Morphological open/close       → clean binary mask
  ↓  Contour filtering (area+shape) → simple car counter    [Method 1]
  ↓  HOG descriptor                 → 1764-dim feature vector
  ↓  LinearSVM (C=best_C)           → patch classifier      [Method 2]
  ↓  Sliding window + NMS           → full-tile detection
  ↓  Count surviving boxes          → car count → submission
```

### Key takeaways

| Question | Answer |
|:--------:|:------:|
| Why preprocess? | Noise and contrast differences break thresholding |
| Why morphology? | Removes isolated pixels that confuse contour detection |
| Why HOG? | Illumination-invariant gradient descriptor |
| Why SVM? | Maximum-margin classifier, works well in high dimensions |
| Why NMS? | Sliding window produces duplicate detections |
| YOLO vs this | YOLO learns all steps end-to-end; classical separates them manually |

### When to use classical CV:
- ✅ Limited data (no annotations needed for thresholding/contours)
- ✅ Interpretable pipeline — you can debug each step
- ✅ Low compute — runs on CPU in real time
- ❌ Complex scenes, occlusion, large appearance variation → use DL

### Further reading:
- Dalal & Triggs (2005) — original HOG paper
- [OpenCV docs — Feature detection](https://docs.opencv.org/4.x/d7/d8b/tutorial_py_feature_homography.html)
- [scikit-image HOG](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.hog)
